# TextRNN

Char-level LSTM: корпус, следующий символ, сэмплинг.
В конце модель уходит в `.pt`, этот файл отдаётся боту.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from kizaru.checkpoint import save_model
from kizaru.data import get_batch, load_corpus, text_to_seq
from kizaru.model import TextRNN
from kizaru.sample import evaluate

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
text_sample = load_corpus()
sequence, char_to_idx, idx_to_char = text_to_seq(text_sample)
print(len(text_sample), "chars", len(char_to_idx), "unique")

In [ ]:
SEQ_LEN = 256
BATCH_SIZE = 16

model = TextRNN(
    input_size=len(idx_to_char),
    hidden_size=128,
    embedding_size=128,
    n_layers=2,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

n_epochs = 500
loss_avg = []

for epoch in range(n_epochs):
    model.train()
    train, target = get_batch(sequence, seq_len=SEQ_LEN, batch_size=BATCH_SIZE)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(BATCH_SIZE, device=device)
    output, hidden = model(train, hidden)
    loss = criterion(output.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_avg.append(loss.item())
    if len(loss_avg) >= 50:
        mean_loss = float(np.mean(loss_avg))
        print(f"Loss: {mean_loss:.6f}")
        scheduler.step(mean_loss)
        loss_avg = []
        print(evaluate(
            model, char_to_idx, idx_to_char,
            start_text="Я", prediction_len=200, temp=0.3, device=device, max_lines=6,
        ))

In [ ]:
model_path = ROOT / "checkpoints" / "kizaru.pt"
save_model(model_path, model, char_to_idx, idx_to_char, epoch=n_epochs)
print(model_path)
print("python bot.py", model_path)